# Day-6: Temperature and Top-p Experiments

Start the FastAPI service in a separate terminal with `uvicorn src.main:app --reload`. This notebook sends requests to `http://127.0.0.1:8000/chat`. It records actual provider replies; it does not fabricate results.

Use the same prompt and `max_tokens` for all runs. In Experiment A keep `top_p=1.0`; in Experiment B keep `temperature=0.7`.

In [1]:
import requests
import pandas as pd

API_URL = "http://127.0.0.1:8000/chat"
PROMPT = "Explain vector databases to a backend developer in exactly 3 bullet points."
MAX_TOKENS = 300

def run_chat(temperature: float, top_p: float) -> dict:
    payload = {
        "message": PROMPT,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": MAX_TOKENS,
    }
    response = requests.post(API_URL, json=payload, timeout=90)
    response.raise_for_status()
    return response.json()

## Experiment A — Temperature
Keep `top_p=1.0` fixed and change temperature only.

In [ ]:
temperature_results = []
for value in [0.0, 0.3, 0.7, 1.2]:
    result = run_chat(temperature=value, top_p=1.0)
    temperature_results.append({
        "temperature": value,
        "top_p": 1.0,
        "model": result["model"],
        "reply": result["reply"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "latency_ms": result["latency_ms"],
    })
temperature_df = pd.DataFrame(temperature_results)
display(temperature_df)

## Experiment B — Top-p
Keep `temperature=0.7` fixed and change top-p only.

In [ ]:
top_p_results = []
for value in [0.1, 0.5, 0.9, 1.0]:
    result = run_chat(temperature=0.7, top_p=value)
    top_p_results.append({
        "temperature": 0.7,
        "top_p": value,
        "model": result["model"],
        "reply": result["reply"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "latency_ms": result["latency_ms"],
    })
top_p_df = pd.DataFrame(top_p_results)
display(top_p_df)

## Save actual results
Run the cells above successfully before exporting. The CSV files contain the real responses returned by your configured provider.

In [ ]:
temperature_df.to_csv("temperature_results.csv", index=False)
top_p_df.to_csv("top_p_results.csv", index=False)
print("Saved temperature_results.csv and top_p_results.csv")